<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/16-agent-aliases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent Aliases: Canary Rollouts and Tenant Routing

Every notebook so far has created sessions with `POST /v2/agents/{agent_key}/sessions`, so the calling application is hard-wired to one agent. Changing the agent's prompt, model, or tools then means either editing that agent in place (every user gets the change at once) or creating a new agent and changing every caller to use its key.

An **agent alias** is a stable name that sits in front of one or more agents. Callers create sessions through the alias (`POST /v2/agent_aliases/{alias_key}/sessions`), and the alias's **routing policy** decides which agent owns each new session. Changing the policy is a single API call and takes effect for new sessions immediately; callers don't change.

In this notebook you will:

1. Create two versions of a support agent, `v1` and `v2`
2. Put an alias in front of `v1` and chat through it
3. Canary `v2` to about 10% of accounts with a **weighted** target
4. Route one tenant to `v2` with a **`match`** rule
5. Promote `v2` and see that existing sessions keep their agent
6. Retire `v1`, and see that the API refuses to delete an agent an alias still routes to

Docs: [Agent aliases](https://docs.vectara.com/docs/agents/aliases), [Route sessions with an alias](https://docs.vectara.com/docs/agents/alias-routing), [Tutorial: Canary an agent version](https://docs.vectara.com/docs/tutorials/canary-an-agent-version).

## Getting Started

All you need for this notebook is a `VECTARA_API_KEY` with the `agent_developer` or `agent_administrator` role (or an account `administrator`/`owner` role).

This notebook is self-contained: it creates and deletes its own two agents and one alias, and does not use the corpora from earlier notebooks.

## Setup

In [1]:
import os
import json
from collections import Counter

import requests

api_key = os.environ['VECTARA_API_KEY']

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

In [2]:
# Load the shared helpers. vectara_utils.py sits next to this notebook in the repo;
# Colab fetches it on first run so the same notebook works locally and in Colab.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)

## Step 1: Create Two Versions of an Agent

An alias routes to agents by **key**, so we give both agents explicit keys instead of letting the platform generate them.

`v2` differs from `v1` in one visible way: it ends every answer with a `TL;DR:` line. That makes it easy to tell from a response which version handled a session. In a real rollout the difference might be a new model, a rewritten prompt, or another tool.

The first cell deletes the alias if it was left over from a previous run. An agent can't be deleted while an alias still routes to it (you'll see that error in Step 6), so the alias has to go first.

In [3]:
ALIAS_KEY = "tutorial-support"
V1_KEY = "tutorial-support-v1"
V2_KEY = "tutorial-support-v2"

# Remove the alias from a previous run so the agents below can be recreated.
response = requests.delete(f"{BASE_URL}/agent_aliases/{ALIAS_KEY}", headers=headers)
print(f"Delete leftover alias: {response.status_code}")  # 204 if it existed, 404 otherwise

BASE_INSTRUCTIONS = (
    "You are a customer support agent for Acme Cloud, a file storage service. "
    "Answer in two or three sentences. Free accounts get 5 GB, Pro accounts get 2 TB. "
    "Files in the trash are kept for 30 days."
)


def support_agent(key, name, extra_instructions=""):
    return {
        "key": key,
        "name": name,
        "description": "Acme Cloud support agent (alias tutorial)",
        "model": {"name": "gpt-4o"},
        "first_step_name": "main",
        "steps": {
            "main": {
                "instructions": [{
                    "type": "inline",
                    "name": "support_instructions",
                    "template": BASE_INSTRUCTIONS + extra_instructions,
                }],
                "output_parser": {"type": "default"},
            }
        },
        "tool_configurations": {},
    }


V1_NAME, V2_NAME = "Tutorial Support v1", "Tutorial Support v2"
delete_and_create_agent(support_agent(V1_KEY, V1_NAME), V1_NAME)
_ = delete_and_create_agent(
    support_agent(V2_KEY, V2_NAME, " End every answer with a final line that starts with 'TL;DR:'."),
    V2_NAME,
)

Delete leftover alias: 404


Created agent 'Tutorial Support v1' (key: tutorial-support-v1)


Created agent 'Tutorial Support v2' (key: tutorial-support-v2)


## Step 2: Create an Alias in Front of `v1`

A policy is a list of **rules**, evaluated top to bottom. Each rule has an optional `match` expression and a `targets` shape. A rule without `match` is a catch-all and must be last.

The simplest policy is one catch-all rule with a `single` target: every session goes to `v1`. That behaves exactly like calling `v1` directly, but callers now use the name `tutorial-support`.

In [4]:
def single(agent_key):
    return {"type": "routed", "rules": [{"targets": {"type": "single", "agent_key": agent_key}}]}


response = requests.post(
    f"{BASE_URL}/agent_aliases",
    headers=headers,
    json={
        "key": ALIAS_KEY,
        "name": "Tutorial Support",
        "description": "Stable handle for the Acme Cloud support agent",
        "policy": single(V1_KEY),
    },
)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status: 201
{
  "key": "tutorial-support",
  "name": "Tutorial Support",
  "description": "Stable handle for the Acme Cloud support agent",
  "policy": {
    "type": "routed",
    "rules": [
      {
        "targets": {
          "type": "single",
          "agent_key": "tutorial-support-v1"
        }
      }
    ]
  },
  "enabled": true,
  "metadata": {},
  "created_at": "2026-10-06T17:52:38.000Z"
}


## Step 3: Create a Session Through the Alias and Chat

The session-create response includes both `alias_key` and the resolved `agent_key`. After that, the session behaves like any other: you post input to its events endpoint, here through the alias path `/v2/agent_aliases/{alias_key}/sessions/{session_key}/events`.

In [5]:
def create_alias_session(metadata=None):
    """Create a session through the alias; return (session_key, agent_key).

    No session name is passed: names must be unique per agent, and the
    platform generates one when it's omitted.
    """
    response = requests.post(
        f"{BASE_URL}/agent_aliases/{ALIAS_KEY}/sessions",
        headers=headers,
        json={"metadata": metadata or {}},
    )
    response.raise_for_status()
    session = response.json()
    return session["key"], session["agent_key"]


def chat(session_key, message):
    """Send a message through the alias and return the agent's reply."""
    response = requests.post(
        f"{BASE_URL}/agent_aliases/{ALIAS_KEY}/sessions/{session_key}/events",
        headers=headers,
        json={"type": "input_message", "messages": [{"type": "text", "content": message}]},
    )
    response.raise_for_status()
    for event in response.json().get("events", []):
        if event["type"] == "error":
            return f"[error] {event.get('messages')}"
        if event["type"] == "agent_output":
            return event["content"]
    return "[no agent_output]"


first_session, agent_key = create_alias_session({"account_id": "acme"})
print(f"Session {first_session} -> {agent_key}\n")
print(chat(first_session, "How long do deleted files stay in the trash?"))

Session ase_d0b1_abbff894a2234fe60d24 -> tutorial-support-v1



Deleted files remain in the trash for 30 days before they are permanently deleted from your Acme Cloud account.


## Step 4: Canary `v2` with a Weighted Target

To send a small share of traffic to `v2`, replace the policy with a `weighted` target. Weights are relative, so `90`/`10` sends roughly 10% of sessions to `v2`.

`partition_by` is the value the split is hashed on. Partitioning by `account_id` keeps each account on one version for the whole canary, rather than switching between versions from one session to the next. It must not evaluate to `null`, so we give `get()` a default of `''` for sessions without an `account_id`.

`PUT /v2/agent_aliases/{alias_key}/policy` replaces the whole policy at once, and only affects sessions created afterwards.

In [6]:
def canary(stable_key, canary_key, canary_weight):
    return {
        "type": "routed",
        "rules": [{
            "targets": {
                "type": "weighted",
                "partition_by": "get('$.session.metadata.account_id', '')",
                "options": [
                    {"agent_key": stable_key, "weight": 100 - canary_weight},
                    {"agent_key": canary_key, "weight": canary_weight},
                ],
            }
        }],
    }


def set_policy(policy):
    response = requests.put(
        f"{BASE_URL}/agent_aliases/{ALIAS_KEY}/policy", headers=headers, json={"policy": policy}
    )
    response.raise_for_status()


set_policy(canary(V1_KEY, V2_KEY, canary_weight=10))

ACCOUNTS = [f"account-{i:02d}" for i in range(30)]


def route_accounts():
    """Create one session per account and return {account_id: agent_key}."""
    return {acct: create_alias_session({"account_id": acct})[1] for acct in ACCOUNTS}


first_pass = route_accounts()
print("Sessions per agent:", dict(Counter(first_pass.values())))
print("Accounts on v2:", [acct for acct, agent in first_pass.items() if agent == V2_KEY])

Sessions per agent: {'tutorial-support-v1': 25, 'tutorial-support-v2': 5}
Accounts on v2: ['account-05', 'account-08', 'account-14', 'account-17', 'account-23']


Each account is hashed independently, so with only 30 accounts the share on `v2` won't be exactly 10%; it gets closer as traffic grows. More important is that the assignment is **deterministic**: an account that landed on `v2` lands there again on its next session.

In [7]:
second_pass = route_accounts()
print("Same assignment on second pass:", first_pass == second_pass)

Same assignment on second pass: True


To widen the rollout you'd call `set_policy(canary(V1_KEY, V2_KEY, 50))`, and so on. To roll back, set the policy back to `single(V1_KEY)`. Each change is atomic.

## Step 5: Route a Tenant with a `match` Rule

Rules can also route on session metadata. Here, sessions whose `tenant` is `eu` go to `v2`, and everything else falls through to the catch-all on `v1`. `match` is a [UserFn expression](https://docs.vectara.com/docs/search-and-retrieval/rerankers/user-defined-function-reranker) over the session context. A missing path evaluates to `null`, so a session without `tenant` doesn't match and falls through.

In [8]:
set_policy({
    "type": "routed",
    "rules": [
        {
            "match": "get('$.session.metadata.tenant') == 'eu'",
            "targets": {"type": "single", "agent_key": V2_KEY},
        },
        {"targets": {"type": "single", "agent_key": V1_KEY}},
    ],
})

for metadata in [{"tenant": "eu"}, {"tenant": "us"}, {}]:
    session_key, agent_key = create_alias_session(metadata)
    print(f"metadata={json.dumps(metadata):<20} -> {agent_key}")

metadata={"tenant": "eu"}     -> tutorial-support-v2
metadata={"tenant": "us"}     -> tutorial-support-v1


metadata={}                   -> tutorial-support-v1


Each rule can use its own `single` or `weighted` target, so you could canary `v2` inside one tenant while routing every other tenant directly.

## Step 6: Promote `v2`

Promotion is another policy replace, this time to a single target on `v2`. Sessions that already exist keep the agent they were created with: the session from Step 3 still belongs to `v1` and keeps working, while new sessions go to `v2`.

In [9]:
set_policy(single(V2_KEY))

new_session, agent_key = create_alias_session({"account_id": "acme"})
print(f"New session -> {agent_key}")
print(chat(new_session, "How much storage does a Pro account get?"))

response = requests.get(f"{BASE_URL}/agent_aliases/{ALIAS_KEY}/sessions/{first_session}", headers=headers)
response.raise_for_status()
print(f"\nStep 3 session still -> {response.json()['agent_key']}")
print(chat(first_session, "And how much storage does a Free account get?"))

New session -> tutorial-support-v2


A Pro account with Acme Cloud receives 2 TB of storage space. This substantial increase from the free account's 5 GB allows for much larger files and more extensive data storage. 

TL;DR: Pro accounts have 2 TB of storage.

Step 3 session still -> tutorial-support-v1


A Free account with Acme Cloud comes with 5 GB of storage.


## Step 7: Retire `v1`

Before deleting an agent, list the aliases that still route to it with `GET /v2/agent_aliases?aliased_agent_key=...`. After the promotion, nothing routes to `v1`, but the alias still routes to `v2`.

The API enforces this: deleting an agent that an alias still references returns `409`. That protects callers from an alias that suddenly points at nothing.

In [10]:
for agent_key in (V1_KEY, V2_KEY):
    response = requests.get(
        f"{BASE_URL}/agent_aliases", headers=headers, params={"aliased_agent_key": agent_key}
    )
    response.raise_for_status()
    print(f"Aliases routing to {agent_key}: {[a['key'] for a in response.json()['aliases']]}")

response = requests.delete(f"{BASE_URL}/agents/{V2_KEY}", headers=headers)
print(f"\nDELETE {V2_KEY}: {response.status_code} {response.text}")

response = requests.delete(f"{BASE_URL}/agents/{V1_KEY}", headers=headers)
print(f"DELETE {V1_KEY}: {response.status_code}")

Aliases routing to tutorial-support-v1: []


Aliases routing to tutorial-support-v2: ['tutorial-support']



DELETE tutorial-support-v2: 409 {"messages":["Cannot delete agent 'tutorial-support-v2'; referenced by aliases: [tutorial-support]"],"request_id":"b864d19b-f56d-4f92-9486-b2687e9568c4"}


DELETE tutorial-support-v1: 204


## Cleanup (Optional)

Delete the alias first, then the agent it routed to.

In [11]:
response = requests.delete(f"{BASE_URL}/agent_aliases/{ALIAS_KEY}", headers=headers)
print(f"Delete alias {ALIAS_KEY}: {response.status_code}")

response = requests.delete(f"{BASE_URL}/agents/{V2_KEY}", headers=headers)
print(f"Delete agent {V2_KEY}: {response.status_code}")

Delete alias tutorial-support: 204


Delete agent tutorial-support-v2: 204


## Summary

- An **alias** is a stable name callers create sessions through; its **policy** picks the agent for each new session.
- **`single`** targets route directly, **`weighted`** targets split traffic by hashing `partition_by`, and **`match`** rules route on session metadata. Rules are checked in order and the first match wins.
- `PUT /v2/agent_aliases/{alias_key}/policy` replaces the policy atomically. Only new sessions are affected, so canary, rollback, and promotion never disrupt sessions already in progress.
- An agent that an alias still routes to can't be deleted (`409`); find dependents with `aliased_agent_key`.

Aliases are also what Vectara's connectors (Slack, Google Chat, the embeddable widget) and [end-user sessions](https://docs.vectara.com/docs/agents/end-user-sessions) attach to, so putting an alias in front of an agent early leaves room for those later.